# Sesión 9 · Corrida final del pipeline del equipo
El cuaderno trabaja sobre el repositorio del equipo, ya unido: usa la corrida del lote de 20, mide la exactitud por campo contra `insumos/respuestas`, lista la revisión humana, revisa el registro y muestra el tablero. No se pega código: cada celda importa y ejecuta lo que el equipo unió.

## 0. El repositorio del equipo
La primera celda encuentra la carpeta `pipeline-equipo-E0N` y muestra la rama y los insumos (20 + 20).

In [ ]:
# 0. Dónde está el repositorio del equipo: la variable REPO_EQUIPO, la carpeta actual o la única carpeta pipeline-equipo-* de C:\curso\trabajo
import json
import os
import re
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd


def es_repo_equipo(carpeta: Path) -> bool:
    return (carpeta / "mi_pipeline" / "__main__.py").is_file() and (carpeta / "insumos" / "documentos").is_dir()


candidatas = [Path(os.environ["REPO_EQUIPO"])] if os.environ.get("REPO_EQUIPO") else []
candidatas += [Path.cwd(), *Path.cwd().parents, *sorted(Path(r"C:\curso\trabajo").glob("pipeline-equipo-*"))]
REPO = next((c.resolve() for c in candidatas if es_repo_equipo(c)), None)
assert REPO is not None, "No se encuentra el repositorio del equipo con mi_pipeline e insumos: se repite «git switch main» y «git pull» en la carpeta del equipo."
sys.path.insert(0, str(REPO))
ENTORNO = dict(os.environ, PYTHONIOENCODING="utf-8")
SALIDA, RESPUESTAS = REPO / "resultados", REPO / "insumos" / "respuestas"
rama = subprocess.run(["git", "branch", "--show-current"], cwd=REPO, capture_output=True, text=True).stdout.strip()
print("Repositorio del equipo:", REPO)
print("Rama:", rama, "| documentos:", len(list((REPO / "insumos" / "documentos").iterdir())), "| respuestas:", len(list(RESPUESTAS.glob("*.json"))))

## 1. La corrida del lote
Si el líder ya subió la corrida, se usa; si no, se corre aquí (en el flujo B, con OCR, tarda más).

In [ ]:
# 1. La corrida del lote: si el líder ya la subió (resultados/json) o ya está en esta VM, se usa; si no, se corre ahora
hay_corrida = (SALIDA / "resultados.duckdb").is_file() or any((SALIDA / "json").glob("*.json"))
if hay_corrida:
    print("Se usa la corrida que ya está en resultados/ (la del líder o una anterior en esta VM).")
else:
    inicio = time.perf_counter()
    corrida = subprocess.run([sys.executable, "-m", "mi_pipeline", "--entrada", "insumos/documentos", "--salida", "resultados"],
                             cwd=REPO, capture_output=True, text=True, encoding="utf-8", errors="replace", env=ENTORNO)
    print("\n".join(corrida.stdout.strip().splitlines()[:6]))
    print(f"Tiempo de la corrida: {time.perf_counter() - inicio:.0f} s")
    assert corrida.returncode == 0, "El pipeline no terminó: " + corrida.stderr[-800:]
print("En resultados/:", sorted(p.name for p in SALIDA.iterdir()))

## 2. Exactitud por campo
La tabla queda ordenada de menor a mayor exactitud: los primeros campos son los que el equipo explica en la exposición.

In [ ]:
# 2. Exactitud por campo contra insumos/respuestas (el mismo comando de la guía de calidad)
medida = subprocess.run([sys.executable, "-m", "mi_pipeline.calidad.exactitud_equipo", "--salida", "resultados", "--respuestas", "insumos/respuestas"],
                        cwd=REPO, capture_output=True, text=True, encoding="utf-8", errors="replace", env=ENTORNO)
print(medida.stdout)
assert medida.returncode == 0, medida.stderr[-800:]
n_json = len(list((SALIDA / "json").glob("*.json")))
n_docs = len(list((REPO / "insumos" / "documentos").iterdir()))
assert n_json == n_docs, f"Hay {n_json} salidas JSON para {n_docs} documentos: la corrida no procesó todo el lote."
exactitud = pd.read_csv(SALIDA / "exactitud_por_campo.csv", encoding="utf-8-sig")
exactitud.sort_values("exactitud")

## 3. Revisión humana
Cada documento con un campo vacío va a revisión: la lista dice cuáles y por qué.

In [ ]:
# 3. Revisión humana: lo que el pipeline no llenó lo ve una persona (no se inventa)
revision = pd.read_csv(SALIDA / "revision_humana.csv", encoding="utf-8-sig").fillna("")
print(f"{len(revision)} de {n_json} documentos van a revisión humana")
vacios = pd.Series(" ".join(revision["campos_vacios"]).split()).value_counts()
print("Campos que más quedan vacíos:", ", ".join(f"{c} ({n})" for c, n in vacios.head(5).items()) or "ninguno")
revision.head(10)

## 4. Registro sin datos personales
Se buscan en el registro los DNI, nombres, teléfonos, correos y cuentas de las respuestas correctas.

In [ ]:
# 4. Registro sin datos personales: ningún DNI, nombre, teléfono, correo ni cuenta de los documentos aparece en registro.jsonl
registro = (SALIDA / "registro.jsonl").read_text(encoding="utf-8")
eventos = [json.loads(linea) for linea in registro.splitlines() if linea.strip()]
PERSONALES = {"nombre", "dni", "telefono", "correo", "cliente", "dni_cliente", "asegurado", "beneficiario", "firmante", "cuenta_cargo", "cci"}
valores = {str(v) for p in RESPUESTAS.glob("*.json") for k, v in json.loads(p.read_text(encoding="utf-8"))["campos"].items() if k in PERSONALES and v}
fugas = sorted(v for v in valores if v in registro)
patrones = re.findall(r"\b99\d{6}\b|[\w.]+@ejemplo\.pe|999 000 \d{3}", registro)
print(f"Eventos en el registro: {len(eventos)} | datos personales buscados: {len(valores)} | encontrados: {len(fugas) + len(patrones)}")
assert not fugas and not patrones, f"El registro guarda datos personales: {(fugas + patrones)[:5]}"

## 5. Tablero
El tablero de la sesión 7, ahora sobre la corrida del equipo.

In [ ]:
# 5. El tablero de la corrida con más documentos del registro (la del lote de 20)
import mi_pipeline
from mi_pipeline.monitoreo.metricas import alertas, calcular_metricas, tablero, umbrales_de
from mi_pipeline.orquestacion.config import cargar_config

assert Path(mi_pipeline.__file__).resolve().is_relative_to(REPO), f"mi_pipeline se carga desde {mi_pipeline.__file__}, no desde el repositorio del equipo"
por_corrida = pd.Series([e.get("corrida_id") for e in eventos if str(e.get("evento", "")).startswith("documento_")]).value_counts()
metricas = calcular_metricas(SALIDA / "registro.jsonl", por_corrida.index[0])
print(tablero(metricas, alertas(metricas, umbrales_de(cargar_config()))))

## 6. Qué se sube
Con la medida lista, calidad guarda `resultados` desde la terminal (guía de calidad, paso 4).

In [ ]:
# 6. Qué queda listo para subir: los archivos nuevos de resultados/ (la base .duckdb y los Parquet no se suben)
estado = subprocess.run(["git", "status", "--short", "--untracked-files=all", "resultados"], cwd=REPO, capture_output=True, text=True).stdout.splitlines()
print(f"Archivos de resultados/ por guardar: {len(estado)}")
print("\n".join(estado[:8]))
ignorada = subprocess.run(["git", "check-ignore", "-q", "resultados/resultados.duckdb"], cwd=REPO)
assert ignorada.returncode == 0, "La base resultados.duckdb no está en el .gitignore: se revisa el .gitignore que trajo el zip de insumos."